# 08 — PPO con capacidad operativa V2

Entrena un agente Proximal Policy Optimization actor-crítico reproducible con NumPy y compara seis políticas. Se mantienen únicamente las categorías **bajo, medio y alto**; `3 = Teleorientación` es una acción, no otra categoría.

> Simulación metodológica para priorización operativa; no es una recomendación clínica.


In [ ]:
from pathlib import Path
import os, sys, subprocess
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = 'feature/entorno-capacidad-v2'
PROJECT_ROOT = Path('/content/pida-rl-datos-publicos')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'gymnasium', 'pandas'])
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(PROJECT_ROOT)])
else:
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'fetch', 'origin', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'checkout', BRANCH])
    subprocess.check_call(['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only', 'origin', BRANCH])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## Configuración

Las semillas de entrenamiento y evaluación son independientes. Los parámetros de GAE, recorte PPO, entropía y parada por KL quedan explícitos para auditoría.


In [ ]:
CAPACIDAD_MENSUAL = 100
COSTOS_ACCION = {0: 0, 1: 1, 2: 3, 3: 5}
HORIZONTE_MESES = 12
PASOS_ENTRENAMIENTO = 100_000
ROLLOUT_STEPS = 2_048
SEMILLA_ENTRENAMIENTO = 30_000
SEMILLAS_EVALUACION = (42, 123, 2026)
CAPAS_OCULTAS = (64, 64)
GAMMA = 0.95
GAE_LAMBDA = 0.95
CLIP_RATIO = 0.20
LEARNING_RATE = 3e-4
VALUE_COEF = 0.50
ENTROPY_COEF = 0.01
UPDATE_EPOCHS = 10
MINIBATCH_SIZE = 64
MAX_KL = 0.03


## Cohorte procesada

Se reutiliza el archivo público procesado por el notebook 03; este notebook no fabrica pacientes.


In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/PIDA-RL-Diabetes')
COHORT_PATH = ROOT / 'data' / 'processed' / 'cohorte_diabetes.csv'
OUTPUT_DIR = ROOT / 'results' / 'capacidad_v2' / 'ppo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cohorte = pd.read_csv(COHORT_PATH, dtype={'FOLIO_INT': 'string'})
assert set(cohorte['categoria_riesgo'].unique()) <= {'bajo', 'medio', 'alto'}
print(cohorte.shape, cohorte['categoria_riesgo'].value_counts().to_dict())


## Entrenamiento PPO

PPO recolecta trayectorias on-policy, calcula ventajas GAE y optimiza una política categórica y una función de valor mediante el objetivo recortado.


In [ ]:
from src.environment.diabetes_followup_env import DiabetesFollowUpEnv
from src.policies.ppo import entrenar_ppo

def crear_entorno():
    return DiabetesFollowUpEnv(cohorte, CAPACIDAD_MENSUAL, COSTOS_ACCION, HORIZONTE_MESES)

agente_ppo, historial_ppo = entrenar_ppo(
    crear_entorno, PASOS_ENTRENAMIENTO, ROLLOUT_STEPS, SEMILLA_ENTRENAMIENTO,
    CAPAS_OCULTAS, GAMMA, GAE_LAMBDA, CLIP_RATIO, LEARNING_RATE,
    VALUE_COEF, ENTROPY_COEF, UPDATE_EPOCHS, MINIBATCH_SIZE, MAX_KL,
)
historial_ppo.to_csv(OUTPUT_DIR / 'historial_entrenamiento_ppo.csv', index=False)
agente_ppo.guardar(OUTPUT_DIR / 'ppo_v2.npz')
display(historial_ppo.tail())


## Evaluación común

PPO se evalúa de forma determinista con las mismas semillas, capacidad y cohorte. Q-learning y DQN se incorporan si existen sus modelos previos.


In [ ]:
from src.evaluation.evaluate_policies import evaluar_politicas
from src.evaluation.reporting import construir_reporte, exportar_reporte_csv
from src.policies.baselines import POLITICAS_BASE
from src.policies.q_learning import QLearningAgent, crear_politica_q_learning
from src.policies.dqn import DQNAgent, crear_politica_dqn
from src.policies.ppo import crear_politica_ppo

politicas = dict(POLITICAS_BASE)
Q_PATH = ROOT / 'results' / 'capacidad_v2' / 'q_learning' / 'q_learning_v2.npz'
DQN_PATH = ROOT / 'results' / 'capacidad_v2' / 'dqn' / 'dqn_v2.npz'
if Q_PATH.exists():
    politicas['Q-learning'] = crear_politica_q_learning(QLearningAgent.cargar(Q_PATH, seed=20_000))
else:
    print('Aviso: ejecute el notebook 06 para incluir Q-learning.')
if DQN_PATH.exists():
    politicas['DQN'] = crear_politica_dqn(DQNAgent.cargar(DQN_PATH, seed=20_000))
else:
    print('Aviso: ejecute el notebook 07 para incluir DQN.')
politicas['PPO'] = crear_politica_ppo(agente_ppo)
resultados = evaluar_politicas(crear_entorno, politicas, SEMILLAS_EVALUACION)
reporte = construir_reporte(resultados)
exportar_reporte_csv(reporte, OUTPUT_DIR)
detalle = reporte['detalle']
assert set(detalle['categoria_riesgo']) <= {'bajo', 'medio', 'alto'}
assert (detalle['recursos_despues'] >= 0).all()
assert (detalle['recursos_usados_mes'] <= detalle['capacidad_mensual']).all()
display(reporte['resumen_algoritmos'].round(4))
display(reporte['clasificaciones_propuestas'].round(2))
display(reporte['clasificaciones_ejecutadas'].round(2))


## Interpretación responsable

- Evalúe recompensa, eventos, cobertura, recursos, equidad y ajustes de capacidad conjuntamente.
- Distinga siempre las acciones propuestas de las ejecutadas.
- Revise entropía, pérdida de valor, KL y variación entre semillas antes de interpretar PPO.
- PPO aprende la recompensa del simulador; no demuestra efectividad clínica.
- Si cambian cohorte, recompensa, costos o capacidad, todos los agentes deben reentrenarse.
